# 22 · y1-strict audit: one-hop tracing

Inputs: `flow_key_tx`, `flow_tx_inputs`, `flow_tx_outputs`, `dest_key_info`, `exchange_keys`, `exchange_addresses_raw`, `exchange_cospend` (04); `panel_keys`, `panel_event` (05); `placebo_draws_strict`

Outputs: `out22/`, zipped as `out22.zip`

One-hop tracing of y1-strict outputs: transaction pattern and fate, stricter outcomes, and automatic labels for the 200-transaction sample of 04.

In [ ]:
# Configuration
PROJECT_ID = "your-gcp-project"
DATASET    = "qbtc_raw"
LOCATION   = "US"

AUTO_RUN_USD      = 5.0
MAX_USD_PER_QUERY = 10.0
MAX_USD_TOTAL     = 100.0
PRICE_PER_TIB_USD = 6.25
CONFIRM           = False
FORCE_REBUILD     = False

FREEZE_DATE   = "2026-09-22"   # freeze date
POST_MAX      = 30             # outcomes: relative days 0...+30 (as in 05)
DORMANT_DAYS  = 30             # y1 strict: unspent for the next 30 days (as in 04)
CONSOL_MIN_IN = 20             # spending transaction with at least this many inputs -> consolidation
CJ_MIN_KEYS   = 3              # CoinJoin: at least 3 input keys
CJ_MIN_SAME   = 3              #           and at least 3 outputs of equal value

H_MAIN      = 90               # as in 20: local linear bandwidth (days)
BANDWIDTHS  = [60, 90, 120]
POOLS       = ["main", "sym30"]
DELTA2      = 0.0235 / 4       # equivalence margin: 1/4 of the Randstorm effect (21)
REF_POOLED  = 8.184148315025512e-05   # main result of 20: LC, key-weighted, y1 strict, +30, main pool, h = 90
AUDIT_N, SEED = 200, 20260923          # as in step 9 of 04, to reproduce the same 200 transactions
OUT_DIR = "out22"

try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass

from google.cloud import bigquery
import json, os, io, re, time, zipfile, datetime as dt
import numpy as np
import pandas as pd

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
DS  = f"{PROJECT_ID}.{DATASET}"
PUB = "bigquery-public-data.crypto_bitcoin"
os.makedirs(OUT_DIR, exist_ok=True)

LEDGER_PATH, JOBS_PATH = "build_ledger.json", "jobs.json"
LEDGER = json.load(open(LEDGER_PATH)) if os.path.exists(LEDGER_PATH) else {"jobs": [], "usd_total": 0.0}
JOBS   = json.load(open(JOBS_PATH))   if os.path.exists(JOBS_PATH)   else []

for t in ["flow_days", "flow_key_tx", "flow_tx_inputs", "flow_tx_outputs", "dest_key_info", "exchange_keys",
          "exchange_addresses_raw", "exchange_cospend", "utxo_life", "panel_keys", "panel_event", "placebo_draws_strict"]:
    client.get_table(f"{DS}.{t}")        # raises if a table is missing
print("Tables from earlier steps found. Spent in this Colab session: $%.2f / $%.0f" % (LEDGER["usd_total"], MAX_USD_TOTAL))

In [ ]:
# Helpers
def usd(gb): return gb / 1024 * PRICE_PER_TIB_USD

def dry_run(sql):
    job = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    gb = job.total_bytes_processed / 2**30
    return gb, usd(gb)

def table_rows(name):
    try:
        return client.get_table(f"{DS}.{name}").num_rows
    except Exception:
        return None

def table_info(name):
    t = client.get_table(f"{DS}.{name}")
    return {"table": name, "rows": t.num_rows, "GB": round(t.num_bytes / 2**30, 2)}

def build(name, sql, force=False):
    assert not re.search(r"CLUSTER BY[^\n]*\b(key_id|txid|spending_txid)\b", sql), f"[{name}] cannot cluster on a BYTES column"
    n = table_rows(name)
    if n and not (FORCE_REBUILD or force):
        print(f"[{name}] exists, {n:,} rows; skipped (rebuild with build(..., force=True))")
        return True
    gb, cost = dry_run(sql)
    if cost > MAX_USD_PER_QUERY:
        raise RuntimeError(f"[{name}] estimated ${cost:.2f}, above the per-query limit ${MAX_USD_PER_QUERY}")
    if LEDGER["usd_total"] + cost > MAX_USD_TOTAL:
        raise RuntimeError(f"[{name}] total would exceed ${MAX_USD_TOTAL}")
    if cost > AUTO_RUN_USD and not CONFIRM:
        raise RuntimeError(f"[{name}] estimated {gb:,.0f} GB ~ ${cost:.2f}, above the auto-run limit ${AUTO_RUN_USD}: set CONFIRM=True and re-run this cell (tables already built are skipped)")
    print(f"[{name}] estimated {gb:,.0f} GB ~ ${cost:.2f}")
    job = client.query(sql, job_config=bigquery.QueryJobConfig(
        maximum_bytes_billed=int(gb * 1.25 * 2**30) + 10 * 2**30, use_query_cache=False))
    JOBS.append({"table": name, "job_id": job.job_id, "location": LOCATION,
                 "submitted_at": dt.datetime.now(dt.timezone.utc).isoformat()})
    json.dump(JOBS, open(JOBS_PATH, "w"), indent=2)
    print(f"[{name}] submitted job {job.job_id} (runs on the server; finishes even if Colab disconnects)")
    t0 = time.time()
    job.result()
    billed = (job.total_bytes_billed or 0) / 2**30
    LEDGER["jobs"].append({"table": name, "job_id": job.job_id, "billed_gb": round(billed, 1),
                           "usd": round(usd(billed), 2), "seconds": round(time.time() - t0, 1)})
    LEDGER["usd_total"] = round(LEDGER["usd_total"] + usd(billed), 2)
    json.dump(LEDGER, open(LEDGER_PATH, "w"), indent=2)
    print(f"[{name}] done, {billed:,.0f} GB, {LEDGER['jobs'][-1]['seconds']}s", table_info(name))
    return True

def need(*names):
    missing = [n for n in names if not table_rows(n)]
    assert not missing, f"These tables are not built yet: {missing} (run the earlier cells first)"

def q(sql):
    return client.query(sql).to_dataframe()

def q_guarded(sql, label):
    """Queries that do not build a table are also priced first and stop above the auto-run limit."""
    gb, cost = dry_run(sql)
    if cost > MAX_USD_PER_QUERY:
        raise RuntimeError(f"[{label}] estimated ${cost:.2f}, above the per-query limit")
    if cost > AUTO_RUN_USD and not CONFIRM:
        raise RuntimeError(f"[{label}] estimated ${cost:.2f}, above the auto-run limit: set CONFIRM=True")
    print(f"[{label}] estimated {gb:,.1f} GB ~ ${cost:.2f}")
    job = client.query(sql, job_config=bigquery.QueryJobConfig(
        maximum_bytes_billed=int(gb * 1.25 * 2**30) + 10 * 2**30, use_query_cache=False))
    df = job.result().to_dataframe()
    billed = (job.total_bytes_billed or 0) / 2**30
    LEDGER["jobs"].append({"table": label, "job_id": job.job_id, "billed_gb": round(billed, 1), "usd": round(usd(billed), 2)})
    LEDGER["usd_total"] = round(LEDGER["usd_total"] + usd(billed), 2)
    json.dump(LEDGER, open(LEDGER_PATH, "w"), indent=2)
    return df

def save(df, name, **kw):
    df.to_csv(os.path.join(OUT_DIR, name), **kw)

In [ ]:
# Dates and constants
FREEZE = pd.Timestamp(FREEZE_DATE)
F  = FREEZE.strftime("%Y-%m-%d")
F1 = (FREEZE + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
fr = q(f"SELECT MIN(date) AS lo, MAX(date) AS hi FROM `{DS}.flow_days`")
LO, HI = pd.Timestamp(fr.lo.iloc[0]).strftime("%Y-%m-%d"), pd.Timestamp(fr.hi.iloc[0]).strftime("%Y-%m-%d")
assert HI <= F, f"last day of flow_days {HI} is after the freeze date {F}"
tx_cols = {f.name for f in client.get_table(f"{PUB}.transactions").schema}
print(f"Dates tagged in 04: {LO} ... {HI}; freeze date {F}; public transactions table has a block_timestamp_month column: {'block_timestamp_month' in tx_cols}")

## Step 1 · y1-strict outputs, `q22_out`

`flow_tx_outputs` from 04 is re-read and the role of each output (change / exchange / exposed / y1 strict / other unexposed / no key) is re-derived with the rules of 04; y1-strict outputs are kept. Scope: all transactions with y1 strict > 0 in `flow_key_tx` (all dates tagged in 04, from day −14; covers the 200-transaction sample of 04).

Pattern of the creating transaction (assigned in order):

| Pattern | Rule |
|---|---|
| `coinjoin` | inputs from ≥ 3 keys and ≥ 3 outputs of equal value |
| `sweep` | apart from change, all outputs are y1 strict (nothing to an existing address, an exchange or an exposed address) |
| `pay_or_change` | other recipients present; the y1-strict output may be change or a payment |

Check: per key and transaction, share × reconstructed y1-strict total = `flow_key_tx.y1_strict`.

In [ ]:
SQL_Q22_OUT = f"""
CREATE OR REPLACE TABLE `{DS}.q22_out`
PARTITION BY created_date
AS
WITH cand AS (   -- at least one risk-set key has a y1-strict outflow in the transaction
  SELECT DISTINCT txid, date
  FROM `{DS}.flow_key_tx`
  WHERE y1_strict > 0 AND date BETWEEN DATE '{LO}' AND DATE '{HI}'
),
tx_in AS (
  SELECT i.spending_txid AS txid,
         ARRAY_AGG(DISTINCT i.key_id IGNORE NULLS) AS in_keys,
         COUNT(*) AS n_in,
         COUNT(DISTINCT i.key_id) AS n_in_keys
  FROM `{DS}.flow_tx_inputs` i
  JOIN cand c ON c.txid = i.spending_txid AND c.date = i.spent_date
  WHERE i.spent_date BETWEEN DATE '{LO}' AND DATE '{HI}'
  GROUP BY i.spending_txid
),
o AS (   -- classification as in 04 (flow_key_tx)
  SELECT o.txid, o.vout, o.created_date, o.sats, o.type, o.address, o.out_spent_date,
    t.n_in, t.n_in_keys,
    IFNULL(o.dest_key_id IN UNNEST(t.in_keys), FALSE)                               AS is_change,
    ex.key_id IS NOT NULL                                                            AS to_ex,
    (o.type IN ('pubkey', 'multisig', 'witness_v1_taproot')
       OR IFNULL(dk.first_exposed_date <= o.created_date, FALSE))                    AS exposed,
    o.dest_key_id IS NOT NULL                                                        AS keyed,
    IFNULL(o.addr_first_seen_height = o.created_height, FALSE)
      AND o.type IN ('pubkeyhash', 'witness_v0_keyhash', 'scripthash', 'witness_v0_scripthash')
      AND (o.out_spent_date IS NULL
           OR o.out_spent_date > DATE_ADD(o.created_date, INTERVAL {DORMANT_DAYS} DAY))  AS strict_ok
  FROM `{DS}.flow_tx_outputs` o
  JOIN cand c ON c.txid = o.txid AND c.date = o.created_date
  JOIN tx_in t ON t.txid = o.txid
  LEFT JOIN `{DS}.dest_key_info` dk ON dk.key_id = o.dest_key_id
  LEFT JOIN `{DS}.exchange_keys` ex ON ex.key_id = o.dest_key_id
  WHERE o.created_date BETWEEN DATE '{LO}' AND DATE '{HI}'
),
r AS (
  SELECT o.*,
    CASE WHEN is_change THEN 'change'
         WHEN to_ex THEN 'y2'
         WHEN exposed THEN 'y0'
         WHEN keyed AND strict_ok THEN 'y1s'
         WHEN keyed THEN 'y1o'
         ELSE 'oth' END AS role,
    COUNT(*) OVER (PARTITION BY txid, sats) AS n_same_value
  FROM o
),
tx AS (
  SELECT txid,
    COUNT(*)                                  AS n_out,
    COUNTIF(role = 'y1s')                     AS n_y1s_out,
    COUNTIF(role NOT IN ('change', 'y1s'))    AS n_third,
    SUM(IF(role = 'y1s', sats, 0))            AS tx_y1s_sats,
    MAX(n_same_value)                         AS max_same_value
  FROM r
  GROUP BY txid
)
SELECT r.txid, r.vout, r.created_date, r.sats, r.type, r.address, r.out_spent_date,
  r.n_in, r.n_in_keys, tx.n_out, tx.n_y1s_out, tx.n_third, tx.tx_y1s_sats, tx.max_same_value,
  CASE WHEN r.n_in_keys >= {CJ_MIN_KEYS} AND tx.max_same_value >= {CJ_MIN_SAME} THEN 'coinjoin'
       WHEN tx.n_third = 0 THEN 'sweep'
       ELSE 'pay_or_change' END AS pattern
FROM r
JOIN tx USING (txid)
WHERE r.role = 'y1s'
"""
build("q22_out", SQL_Q22_OUT)

chk1 = q(f"""
WITH t AS (SELECT txid, ANY_VALUE(tx_y1s_sats) AS s FROM `{DS}.q22_out` GROUP BY txid)
SELECT COUNT(*) AS n_key_tx,
       COUNTIF(t.txid IS NULL) AS missing_tx,
       SUM(f.y1_strict) / 1e8 AS y1s_btc,
       SUM(ABS(f.y1_strict - f.share * IFNULL(t.s, 0))) / 1e8 AS abs_gap_btc
FROM `{DS}.flow_key_tx` f
LEFT JOIN t USING (txid)
WHERE f.y1_strict > 0 AND f.date BETWEEN DATE '{LO}' AND DATE '{HI}'
""")
display(chk1)
c = chk1.iloc[0]
assert c.missing_tx == 0, "some transactions with y1 strict were not reconstructed"
assert c.abs_gap_btc <= 1e-6 * max(c.y1s_btc, 1), "reconstructed y1 strict does not match 04's flow_key_tx"
print("Step 1 check passed: reconstructed y1 strict matches 04 row by row")
display(q(f"""SELECT pattern, COUNT(*) AS n_out, COUNT(DISTINCT txid) AS n_tx, ROUND(SUM(sats)/1e8, 2) AS btc,
                     COUNTIF(out_spent_date IS NULL) AS n_unspent
              FROM `{DS}.q22_out` GROUP BY pattern ORDER BY btc DESC"""))

## Step 2 · One-hop tracing, `q22_fate`, `q22_spend`

1. `q22_fate`: spending transaction (`spending_txid`) of y1-strict outputs spent before the freeze date, from `utxo_life` (partitions within the dates tagged in 04).
2. `q22_spend_raw`: input count, output count and output addresses of these transactions from the public `transactions` table (one scan).
3. `q22_spend`: whether any output address is an exchange address (PoR addresses of 04, plus `exchange_cospend` addresses belonging to a single exchange).

Fate of each y1-strict output (assigned in order):

| Fate | Rule |
|---|---|
| `unspent` | unspent at the freeze date |
| `to_exchange` | spending transaction pays an exchange address |
| `consolidated` | spending transaction has ≥ 20 inputs |
| `spent_kept` | other spends |

y1 strict already requires 30 days unspent, so consolidations within 30 days are excluded upstream.

Check: outputs spent before the freeze date in step 1 = rows of `q22_fate`; every spending transaction is found in the public table.

In [ ]:
need("q22_out")
SQL_FATE = f"""
CREATE OR REPLACE TABLE `{DS}.q22_fate`
AS
WITH y AS (
  SELECT txid, vout, created_date FROM `{DS}.q22_out` WHERE out_spent_date IS NOT NULL
)
SELECT u.txid, u.vout, u.spending_txid, u.spent_date
FROM `{DS}.utxo_life` u
JOIN y ON y.txid = u.txid AND y.vout = u.vout AND y.created_date = u.created_date
WHERE u.created_date BETWEEN DATE '{LO}' AND DATE '{HI}'
"""
build("q22_fate", SQL_FATE)

sp = q(f"""SELECT MIN(spent_date) AS lo, MAX(spent_date) AS hi, COUNT(*) AS n,
                 COUNT(DISTINCT spending_txid) AS n_spend_tx FROM `{DS}.q22_fate`""")
n_spent = q(f"SELECT COUNTIF(out_spent_date IS NOT NULL) AS n FROM `{DS}.q22_out`").n.iloc[0]
display(sp)
assert sp.n.iloc[0] == n_spent, f"q22_fate has {sp.n.iloc[0]:,} rows != {n_spent:,} spent outputs"
assert pd.Timestamp(sp.hi.iloc[0]) <= FREEZE
S_LO = pd.Timestamp(sp.lo.iloc[0]).strftime("%Y-%m-%d")
m0 = pd.Timestamp(S_LO).replace(day=1).strftime("%Y-%m-%d")
month_filter = (f"AND t.block_timestamp_month BETWEEN DATE '{m0}' AND DATE '{F}'"
                if "block_timestamp_month" in tx_cols else "")

SQL_SPEND_RAW = f"""
CREATE OR REPLACE TABLE `{DS}.q22_spend_raw`
AS
WITH s AS (
  SELECT DISTINCT LOWER(TO_HEX(spending_txid)) AS h FROM `{DS}.q22_fate`
)
SELECT t.hash AS h, t.input_count AS n_in, t.output_count AS n_out,
       ARRAY(SELECT a FROM UNNEST(t.outputs) AS o, UNNEST(o.addresses) AS a) AS addrs
FROM `{PUB}.transactions` t
JOIN s ON s.h = t.hash
WHERE t.block_timestamp >= TIMESTAMP('{S_LO}') AND t.block_timestamp < TIMESTAMP('{F1}') {month_filter}
"""
build("q22_spend_raw", SQL_SPEND_RAW)

SQL_SPEND = f"""
CREATE OR REPLACE TABLE `{DS}.q22_spend`
AS
WITH exa AS (
  SELECT address FROM `{DS}.exchange_addresses_raw`
  UNION DISTINCT
  SELECT address FROM `{DS}.exchange_cospend` WHERE n_ex = 1
),
hit AS (
  SELECT DISTINCT r.h
  FROM `{DS}.q22_spend_raw` r
  CROSS JOIN UNNEST(r.addrs) AS a
  JOIN exa ON exa.address = a
)
SELECT r.h, r.n_in, r.n_out, hit.h IS NOT NULL AS to_ex
FROM `{DS}.q22_spend_raw` r
LEFT JOIN hit ON hit.h = r.h
"""
build("q22_spend", SQL_SPEND)
chk2 = q(f"""
SELECT COUNT(DISTINCT f.spending_txid) AS n_spend_tx,
       COUNT(DISTINCT IF(s.h IS NULL, f.spending_txid, NULL)) AS not_found
FROM `{DS}.q22_fate` f
LEFT JOIN `{DS}.q22_spend` s ON s.h = LOWER(TO_HEX(f.spending_txid))
""")
display(chk2)
assert chk2.not_found.iloc[0] == 0, "some spending transactions were not found in the public transactions table"
print("Step 2 checks passed")

## Step 3 · Outcomes per key × event, aggregated to CEM cells

For each key in `panel_keys` × date (as in 05), cumulative share over days 0…+30, capped at 1:

| Outcome | Definition |
|---|---|
| `v_all` | y1 strict, as in 20 |
| `v_sweep` | whole-transaction sweeps only |
| `v_sweep_kept` | sweeps whose outputs were afterwards untouched or kept in use (not sent to an exchange, not consolidated) |
| `v_unspent` | not a CoinJoin, and untouched at the freeze date |

Aggregated to event × group × CEM cell.

Check: `v_all` by cell equals `panel_event.y1_strict_c` of 05 at the censoring day.

In [ ]:
need("q22_out", "q22_fate", "q22_spend")
FATE_SQL = f"""
  SELECT o.txid, o.vout, o.sats, o.pattern,
    CASE WHEN o.out_spent_date IS NULL THEN 'unspent'
         WHEN s.to_ex THEN 'to_exchange'
         WHEN s.n_in >= {CONSOL_MIN_IN} THEN 'consolidated'
         WHEN s.h IS NULL THEN 'spent_unknown'
         ELSE 'spent_kept' END AS fate
  FROM `{DS}.q22_out` o
  LEFT JOIN `{DS}.q22_fate` f USING (txid, vout)
  LEFT JOIN `{DS}.q22_spend` s ON s.h = LOWER(TO_HEX(f.spending_txid))"""
OUTC = {   # outcome -> y1-strict outputs counted in each transaction
    "v_all":        "TRUE",
    "v_sweep":      "pattern = 'sweep'",
    "v_sweep_kept": "pattern = 'sweep' AND fate IN ('unspent', 'spent_kept')",
    "v_unspent":    "pattern != 'coinjoin' AND fate = 'unspent'",
}
cls_cols = ",\n    ".join(f"SUM(IF({cond}, sats, 0)) AS s_{k}" for k, cond in OUTC.items())
kf_cols  = ",\n    ".join(f"SUM(f.share * c.s_{k}) AS a_{k}" for k in OUTC)
k_cols   = ",\n    ".join(f"LEAST(1.0, IFNULL(kf.a_{k}, 0) / pk.sats) AS {k}" for k in OUTC)
agg_cols = ",\n  ".join(f"SUM({k}) AS {k}" for k in OUTC)

SQL_CELLS = f"""
WITH fd AS (
  SELECT event, rel_day, date FROM `{DS}.flow_days` WHERE rel_day BETWEEN 0 AND {POST_MAX}
),
fo AS ({FATE_SQL}
),
c AS (
  SELECT txid,
    {cls_cols}
  FROM fo GROUP BY txid
),
kf AS (
  SELECT fd.event, f.key_id,
    {kf_cols}
  FROM `{DS}.flow_key_tx` f
  JOIN fd ON fd.date = f.date
  JOIN c ON c.txid = f.txid
  WHERE f.y1_strict > 0 AND f.date BETWEEN DATE '{LO}' AND DATE '{HI}'
  GROUP BY fd.event, f.key_id
),
k AS (
  SELECT pk.event, pk.event_set, pk.event_date, pk.grp, pk.cem_id, pk.sats,
    {k_cols}
  FROM `{DS}.panel_keys` pk
  LEFT JOIN kf ON kf.event = pk.event AND kf.key_id = pk.key_id
)
SELECT event, ANY_VALUE(event_set) AS event_set, ANY_VALUE(event_date) AS event_date, grp, cem_id,
  COUNT(*) AS n_keys, SUM(sats) / 1e8 AS bal_btc,
  {agg_cols}
FROM k
GROUP BY event, grp, cem_id
"""
cells = q_guarded(SQL_CELLS, "q22_cells")
cells["event_date"] = pd.to_datetime(cells.event_date)
print(f"cells: {len(cells):,} rows, {cells.event.nunique()} dates")

pe = q_guarded(f"""
SELECT event, grp, cem_id, SUM(y1_strict_c) AS pe_y1s
FROM `{DS}.panel_event` WHERE rel_day = censor_day
GROUP BY event, grp, cem_id""", "panel_event_check")
j = cells.merge(pe, on=["event", "grp", "cem_id"], how="outer")
gap = (j.v_all.fillna(0) - j.pe_y1s.fillna(0)).abs()
print(f"Reconciliation with panel_event: {len(j):,} cells, max difference {gap.max():.2e}, cells not matching {int((gap > 1e-6).sum())}")
assert (gap <= 1e-6).all(), "v_all does not match 05's panel_event.y1_strict_c"
for k in OUTC:
    if k != "v_all":
        assert (cells[k] <= cells.v_all + 1e-9).all(), f"{k} must not exceed v_all"
cells.to_csv(os.path.join(OUT_DIR, "q22_cells.csv.gz"), index=False, compression="gzip")
print("Step 3 check passed: v_all matches 05 cell by cell")

## Step 4 · Composition of y1 strict

y1-strict BTC (allocated by key share) of risk-set keys on days 0…+30, by date type (main events / placebo dates / positive controls) × group (L / C / T) × pattern × fate. The untouched share depends on the time to the freeze date, so L and C are compared on the same dates.

In [ ]:
SQL_COMP = f"""
WITH fd AS (
  SELECT event, rel_day, date FROM `{DS}.flow_days` WHERE rel_day BETWEEN 0 AND {POST_MAX}
),
fo AS ({FATE_SQL}
),
kt AS (
  SELECT fd.event, f.key_id, f.txid, f.share
  FROM `{DS}.flow_key_tx` f
  JOIN fd ON fd.date = f.date
  WHERE f.y1_strict > 0 AND f.date BETWEEN DATE '{LO}' AND DATE '{HI}'
)
SELECT pk.event, pk.event_set, pk.grp, fo.pattern, fo.fate,
  COUNT(*) AS n_key_out, SUM(kt.share * fo.sats) / 1e8 AS btc
FROM kt
JOIN `{DS}.panel_keys` pk ON pk.event = kt.event AND pk.key_id = kt.key_id
JOIN fo ON fo.txid = kt.txid
GROUP BY pk.event, pk.event_set, pk.grp, fo.pattern, fo.fate
"""
comp = q_guarded(SQL_COMP, "q22_comp")
save(comp, "q22_composition_by_event.csv", index=False)

CLASS = {("sweep", "unspent"): "sweep · unspent", ("sweep", "spent_kept"): "sweep · kept",
         ("sweep", "to_exchange"): "sweep · deposit", ("sweep", "consolidated"): "sweep · deposit",
         ("sweep", "spent_unknown"): "sweep · kept"}
comp["cls"] = [CLASS.get((p, f), p) for p, f in zip(comp.pattern, comp.fate)]
comp["cls"] = comp.cls.replace({"pay_or_change": "pay or change", "coinjoin": "coinjoin"})
CLS_ORDER = ["sweep · unspent", "sweep · kept", "sweep · deposit", "pay or change", "coinjoin"]
SETS = {"main": "Quantum events", "placebo": "Placebo dates", "positive_control": "Positive controls"}
tab = (comp[comp.event_set.isin(SETS)]
       .pivot_table(index=["event_set", "grp"], columns="cls", values="btc", aggfunc="sum", fill_value=0)
       .reindex(columns=CLS_ORDER, fill_value=0))
share = tab.div(tab.sum(axis=1), axis=0)
tab_n = (comp[comp.event_set.isin(SETS)]
         .pivot_table(index=["event_set", "grp"], columns="cls", values="n_key_out", aggfunc="sum", fill_value=0)
         .reindex(columns=CLS_ORDER, fill_value=0))
share_n = tab_n.div(tab_n.sum(axis=1), axis=0)
out = share.add_prefix("btc_share: ").join(share_n.add_prefix("count_share: ")).join(tab.sum(axis=1).rename("y1s_btc"))
save(out.reset_index(), "q22_composition.csv", index=False)
print("Composition by BTC:"); display((share * 100).round(1))
print("Composition by number of key x output pairs:"); display((share_n * 100).round(1))

# differences L vs C and event vs placebo (percentage points)
diffs = []
for es in ["main", "placebo"]:
    if (es, "L") in share.index and (es, "C") in share.index:
        diffs.append((share.loc[(es, "L")] - share.loc[(es, "C")]).rename(f"{es}: L − C"))
for g in ["L", "C"]:
    if ("main", g) in share.index and ("placebo", g) in share.index:
        diffs.append((share.loc[("main", g)] - share.loc[("placebo", g)]).rename(f"{g}: event − placebo"))
comp_diff = pd.DataFrame(diffs) * 100
save(comp_diff.reset_index().rename(columns={"index": "comparison"}), "q22_composition_diff.csv", index=False)
print("Composition differences (percentage points, by BTC):"); display(comp_diff.round(1))

import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "sans-serif", "font.size": 8, "axes.spines.top": False, "axes.spines.right": False})
COL = {"sweep · unspent": "#2a78d6", "sweep · kept": "#1baf7a", "sweep · deposit": "#eda100",
       "pay or change": "#eb6834", "coinjoin": "#4a3aa7"}
rows = [(es, g) for es in ["main", "placebo", "positive_control"] for g in ["L", "C", "T"] if (es, g) in share.index]
fig, ax = plt.subplots(figsize=(7.2, 0.35 * len(rows) + 1.2))
for i, r in enumerate(rows[::-1]):
    left = 0
    for c_ in CLS_ORDER:
        v = share.loc[r, c_]
        ax.barh(i, v, left=left, color=COL[c_], height=0.62, edgecolor="white", linewidth=0.8)
        if v > 0.07:
            ax.text(left + v / 2, i, f"{v*100:.0f}%", ha="center", va="center", color="white", fontsize=7, fontweight="bold")
        left += v
ax.set_yticks(range(len(rows))); ax.set_yticklabels([f"{SETS[es]} · {g}" for es, g in rows[::-1]])
ax.set_xlim(0, 1); ax.set_xlabel("Share of y1-strict BTC (key-share weighted, days 0–30)")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=COL[c_]) for c_ in CLS_ORDER], labels=CLS_ORDER,
          loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=5, frameon=False, fontsize=7)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "fig_22_composition.png"), dpi=200); plt.show()

## Step 5 · Pooled estimate of 20 on stricter outcomes

Estimator of 20, LC contrast, key-weighted, all keys: θ, local linear baseline (h = 90; 60 and 120), leave-one-out randomization inference, pooled test against the 1,000 draws of `placebo_draws_strict`, 90% interval = β − 95%/5% residual quantiles. Equivalence margins: δ1 (counterfactual baseline rate) and δ2 = 1/4 of the Randstorm effect (0.59 pp), applied to all outcomes; δ2 may be loose for the stricter outcomes, so the 90% interval is reported alongside.

Regression check: the pooled effect of `v_all` equals `est_20_pooled.csv`.

In [ ]:
def theta_lc(col):
    c = cells[cells.grp.isin(["L", "C"])]
    p = c.pivot_table(index=["event", "cem_id"], columns="grp", values=["n_keys", col], aggfunc="sum", fill_value=0)
    nL, nC = p[("n_keys", "L")], p[("n_keys", "C")]
    sL, sC = p[(col, "L")], p[(col, "C")]
    m = (nL > 0) & (nC > 0)
    g = pd.DataFrame({"den": nL.where(m, 0), "T": sL.where(m, 0),
                      "C": (nL / nC.where(nC > 0)).fillna(0).where(m, 0) * sC}).groupby(level="event").sum()
    return (g["T"] - g["C"]) / g["den"], g["T"] / g["den"]

ev = (cells.groupby("event").agg(event_set=("event_set", "first"), event_date=("event_date", "first"))
           .reset_index().sort_values("event_date").reset_index(drop=True))
T0 = pd.Timestamp("2020-01-01")
tday = ((ev.event_date - T0).dt.days).to_numpy(float)
EV_IDX = {e: i for i, e in enumerate(ev.event)}
TH, LV = {}, {}
for k in OUTC:
    d_, l_ = theta_lc(k)
    TH[k], LV[k] = d_.reindex(ev.event), l_.reindex(ev.event)
diff = pd.DataFrame(TH).T[ev.event.tolist()]      # rows = outcomes, columns = dates
lev  = pd.DataFrame(LV).T[ev.event.tolist()]

draws_all = q(f"SELECT pool, draw_id, slot, fake_date FROM `{DS}.placebo_draws_strict`")
draws_all["fake_date"] = pd.to_datetime(draws_all.fake_date)
pl = ev[ev.event_set == "placebo"].set_index("event_date").event
POOL_EVENTS, DRAWS = {}, {}
for pool in POOLS:
    d = draws_all[draws_all.pool == pool]
    dates = sorted(d.fake_date.unique())
    missing = [x for x in dates if x not in pl.index]
    assert not missing, f"dates of pool {pool} not in cells: {missing}"
    POOL_EVENTS[pool] = [pl[x] for x in dates]
    pos = {x: j for j, x in enumerate(dates)}
    D = np.zeros((d.draw_id.nunique(), len(dates)))
    for r in d.itertuples():
        D[r.draw_id - 1, pos[r.fake_date]] += 1
    DRAWS[pool] = D / D.sum(1, keepdims=True)
    print(f"pool {pool}: {len(dates)} dates, {len(D)} draws")

# helpers as in 20
def smoother(pool, h, mask=None):
    idx = np.array([EV_IDX[e] for e in POOL_EVENTS[pool]])
    tp = tday[idx]
    ok = np.ones(len(idx), bool) if mask is None else mask
    lo = tp[ok].min() - 60
    S = np.zeros((len(ev), len(idx)))
    for i in range(len(ev)):
        w = np.exp(-0.5 * ((tp - tday[i]) / h) ** 2) * ok
        self_ = idx == i
        w[self_] = 0
        if tday[i] < lo:                           # before the pool: pool mean
            w = ok.astype(float); w[self_] = 0
            S[i] = w / w.sum(); continue
        X = np.column_stack([np.ones_like(tp), tp - tday[i]])
        A = X.T @ (X * w[:, None])
        S[i] = np.linalg.solve(A, (X * w[:, None]).T)[0]
    return S

def effects(Th, pool, h):
    idx = [EV_IDX[e] for e in POOL_EVENTS[pool]]
    Y = Th.to_numpy(float)
    P = Y[:, idx]
    B = np.full_like(Y, np.nan)
    nanmask = np.isnan(P)
    keys = pd.Series([m.tobytes() for m in nanmask])
    for key, rows_ in keys.groupby(keys).groups.items():
        rows_ = np.array(list(rows_))
        ok = ~nanmask[rows_[0]]
        if ok.sum() < 5:
            continue
        S = smoother(pool, h, ok)
        B[rows_] = Y[rows_] - np.nan_to_num(P[rows_]) @ S.T
    R = B[:, idx]
    return pd.DataFrame(B, index=Th.index, columns=Th.columns), R

def ri_p(beta, R):
    b = np.abs(np.asarray(beta, float))[:, :, None]
    r = np.abs(R)[:, None, :]
    cnt = np.sum(r >= b - 1e-15, axis=2)
    n = np.sum(~np.isnan(R), axis=1)[:, None]
    p = (1 + cnt) / (1 + n)
    p[np.isnan(np.asarray(beta, float))] = np.nan
    return p

def ri_ci(beta, R, a=0.10):
    lo_q, hi_q = np.nanquantile(R, 1 - a / 2, axis=1), np.nanquantile(R, a / 2, axis=1)
    return beta - lo_q[:, None], beta - hi_q[:, None]

MAIN = ev[ev.event_set == "main"].event.tolist()
REPORT = ev[ev.event_set.isin(["main", "secondary", "nonquantum_news", "positive_control"])].event.tolist()
print("Main events:", MAIN)
eff_rows, pooled_rows = [], []
for pool in POOLS:
    for h in BANDWIDTHS:
        B, R = effects(diff, pool, h)
        p = ri_p(B[REPORT].to_numpy(), R)
        lo, hi = ri_ci(B[REPORT].to_numpy(), R)
        for j_, e in enumerate(REPORT):
            eff_rows.append(pd.DataFrame({"outcome": diff.index, "event": e, "pool": pool, "h": h,
                                          "theta": diff[e].values, "beta": B[e].values, "p_ri": p[:, j_],
                                          "ci90_lo": lo[:, j_], "ci90_hi": hi[:, j_], "level_T": lev[e].values}))
        mb = B[MAIN].mean(axis=1).to_numpy()
        null = np.nan_to_num(R) @ DRAWS[pool].T
        pp = (1 + np.sum(np.abs(null) >= np.abs(mb)[:, None] - 1e-15, axis=1)) / (1 + null.shape[1])
        q05, q95 = np.quantile(null, 0.05, axis=1), np.quantile(null, 0.95, axis=1)
        lvl = lev[MAIN].mean(axis=1).to_numpy()
        pooled_rows.append(pd.DataFrame({"outcome": diff.index, "pool": pool, "h": h, "beta_mean": mb, "p_ri": pp,
                                         "ci90_lo": mb - q95, "ci90_hi": mb - q05, "level_T_mean": lvl}))
eff22 = pd.concat(eff_rows, ignore_index=True)
pooled22 = pd.concat(pooled_rows, ignore_index=True)
pooled22["delta1"] = (pooled22.level_T_mean - pooled22.beta_mean).abs()
pooled22["delta2"] = DELTA2
for dn in ["delta1", "delta2"]:
    pooled22[f"equiv_{dn}"] = (pooled22.ci90_lo > -pooled22[dn]) & (pooled22.ci90_hi < pooled22[dn])
eff22 = eff22.merge(ev[["event", "event_set", "event_date"]], on="event", how="left")
save(eff22, "q22_effects.csv", index=False)
save(pooled22, "q22_pooled.csv", index=False)

ref = pooled22[(pooled22.outcome == "v_all") & (pooled22.pool == "main") & (pooled22.h == H_MAIN)].beta_mean.iloc[0]
print(f"Reproduction check: pooled effect of v_all {ref:.10f}, main result of 20 {REF_POOLED:.10f}")
assert abs(ref - REF_POOLED) < 1e-8, "pooled effect of v_all does not match 20; check step 3"
print("Reproduction passed: this code matches 20 exactly")

show = pooled22[(pooled22.pool == "main") & (pooled22.h == H_MAIN)].set_index("outcome")
display((show[["beta_mean", "ci90_lo", "ci90_hi", "level_T_mean"]] * 100).round(4).join(
        show[["p_ri", "equiv_delta1", "equiv_delta2"]]))
print("(first four columns in percentage points)")
e30 = eff22[(eff22.pool == "main") & (eff22.h == H_MAIN) & eff22.event.isin(MAIN + ["randstorm"])]
display((e30.pivot(index="event", columns="outcome", values="beta") * 100).round(4))
print("beta by event (percentage points); the randstorm row is the coarse L-C contrast, not 21's uncompressed vs compressed")

## Step 6 · Automatic labels for the 200-transaction sample

Same draw as step 9 of 04 (y1-strict transactions in the main windows −14…+30, ordered by `FARM_FINGERPRINT(txid + seed)`, first 200), joined to the patterns and fates of steps 1–2:

| Automatic label | Rule | Manual label |
|---|---|---|
| `self_migration` | sweep, all y1-strict outputs afterwards untouched or kept in use | self_migration |
| `exchange_deposit` | a y1-strict output afterwards sent to an exchange or consolidated | exchange_deposit |
| `pay_or_change` | other recipients present | change_to_new_address or payment |
| `coinjoin` | CoinJoin | coinjoin |

If `y1_audit_sample.csv` from 04 is present, the cell checks that both draws contain the same txids.

In [ ]:
need("q22_out", "q22_fate", "q22_spend")
SQL_AUDIT = f"""
WITH cand AS (
  SELECT f.txid, f.key_id, f.date, f.share, f.key_in_sats, f.y1_strict
  FROM `{DS}.flow_key_tx` f
  WHERE f.y1_strict > 0
    AND f.date IN (SELECT date FROM `{DS}.flow_days` WHERE event_set = 'main' AND in_main_window)
  QUALIFY ROW_NUMBER() OVER (PARTITION BY f.txid ORDER BY f.key_in_sats DESC) = 1
),
pick AS (
  SELECT * FROM cand ORDER BY FARM_FINGERPRINT(CONCAT(TO_HEX(txid), '{SEED}')) LIMIT {AUDIT_N}
),
fo AS ({FATE_SQL}
)
SELECT LOWER(TO_HEX(p.txid)) AS txid, p.date, ROUND(p.share, 4) AS share,
  ROUND(p.y1_strict / 1e8, 8) AS y1_strict_btc,
  ANY_VALUE(fo.pattern) AS pattern,
  COUNT(fo.vout) AS n_y1s_out,
  STRING_AGG(CONCAT(CAST(fo.vout AS STRING), ':', fo.fate), ' | ' ORDER BY fo.vout) AS fates,
  LOGICAL_OR(fo.fate IN ('to_exchange', 'consolidated')) AS any_deposit
FROM pick p
LEFT JOIN fo ON fo.txid = p.txid
GROUP BY 1, 2, 3, 4
ORDER BY p.date
"""
audit = q_guarded(SQL_AUDIT, "q22_audit")
audit["auto_label"] = np.select(
    [audit.pattern == "coinjoin", audit.any_deposit, audit.pattern == "sweep", audit.pattern == "pay_or_change"],
    ["coinjoin", "exchange_deposit", "self_migration", "pay_or_change"], default="unknown")
audit["url"] = "https://mempool.space/tx/" + audit.txid
save(audit, "y1_audit_auto.csv", index=False)
print(f"{len(audit)} transactions; automatic labels:"); display(audit.auto_label.value_counts())
if os.path.exists("y1_audit_sample.csv"):
    old = pd.read_csv("y1_audit_sample.csv")
    same = set(old.txid.str.lower()) == set(audit.txid)
    print("Same txids as 04's y1_audit_sample.csv:", same)
else:
    print("y1_audit_sample.csv not uploaded; txid check skipped")

In [ ]:
for rec in JOBS[-4:]:
    job = client.get_job(rec["job_id"], location=rec.get("location", LOCATION))
    print(rec["table"], job.state, "errors:", job.errors)
    if job.state != "DONE":
        job.result(); print("  done")
for t in ["q22_out", "q22_fate", "q22_spend_raw", "q22_spend"]:
    n = table_rows(t)
    print(f"{t:10s}", f"{n:,} rows" if n else "not built yet")

In [ ]:
report = {"generated_at": dt.datetime.now(dt.timezone.utc).isoformat(),
          "config": {"freeze": F, "flow_days": [LO, HI], "post_max": POST_MAX, "dormant_days": DORMANT_DAYS,
                     "consol_min_in": CONSOL_MIN_IN, "cj_min_keys": CJ_MIN_KEYS, "cj_min_same": CJ_MIN_SAME,
                     "outcomes": OUTC, "h_main": H_MAIN, "bandwidths": BANDWIDTHS, "pools": POOLS,
                     "delta2": DELTA2, "ref_pooled_20": REF_POOLED, "audit_n": AUDIT_N, "seed": SEED},
          "ledger": LEDGER,
          "tables": [table_info(t) for t in ["q22_out", "q22_fate", "q22_spend_raw", "q22_spend"] if table_rows(t)]}
for k, v in [("check_step1", "chk1"), ("check_step2", "chk2"), ("composition", "out.reset_index()"),
             ("composition_diff", "comp_diff.reset_index()"),
             ("pooled_main_h90", "pooled22[(pooled22.pool == 'main') & (pooled22.h == H_MAIN)]"),
             ("audit_labels", "audit.auto_label.value_counts().reset_index()")]:
    try: report[k] = eval(v).astype(str).to_dict(orient="records")
    except NameError: pass
with open(os.path.join(OUT_DIR, "build_report_22.json"), "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2, default=str)
with zipfile.ZipFile("out22.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for fn in sorted(os.listdir(OUT_DIR)):
        z.write(os.path.join(OUT_DIR, fn), arcname=f"out22/{fn}")
print("Packed:", sorted(os.listdir(OUT_DIR)))
try:
    from google.colab import files
    files.download("out22.zip")
except Exception:
    print("out22.zip is in the working directory")